# Bernini-R 图像 / 视频编辑 — Colab G4 启动脚本

**模型**：字节跳动 Bernini-R（基于 Wan2.2 的 renderer-only 上下文条件模型）　**运行时**：G4 高 RAM（96GB）

不需要训 LoRA、不需要微调，靠参考图 / 参考视频作为上下文 token 引导生成。支持下列任务：

| 任务 | 输入 | 说明 |
| --- | --- | --- |
| `t2v` | 文本 | 文生视频 |
| `v2v` | 源视频 | 视频风格化 |
| `rv2v` | 源视频 + 参考图 | 参考引导的视频编辑（重打光、主体插入） |
| `r2v` | 仅参考图 | 参考图生视频 |
| `img` | 源图 + 提示词 | 图像编辑（重打光、改风格、插主体） |
| `ads2v` | 源视频 + 参考视频 | 把图/视频内容插入源视频 |

全程用 ComfyUI **原生节点**，不需要自定义节点。

| 项 | 值 |
| --- | --- |
| 模型数量 | 4 个（主模型 / 文本编码器 / VAE / 加速 LoRA） |
| 内网穿透 | FRP tcp 无 token，端口 **8092** |
| 访问地址 | http://usoren.usdream.dpdns.org:8092 |

按顺序跑 Cell 1 → 5。重启界面只需重跑 Cell 4 与 Cell 5。

> **端口说明**：本笔记本用 8092，SCAIL-2 笔记本用 8091，两者可同时跑在不同 Colab 会话里互不冲突。

> **服务端前提**：frps 配置里不能有 `auth.token`；8092 需空闲且在 `allowPorts` 内。


In [ ]:
# ==========================================
# Cell 1: 安装 ComfyUI (必须是最新 nightly, Bernini-R 节点才存在)
# ==========================================
import os
import subprocess

print("=== 🚀 安装 ComfyUI ===")
%cd /content

if not os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI
else:
    !cd ComfyUI && git pull

%cd /content/ComfyUI
!pip install -q -r requirements.txt huggingface_hub hf_transfer

# ComfyUI-Manager 只作排错用, 本工作流不需要任何自定义节点
MGR = "/content/ComfyUI/custom_nodes/ComfyUI-Manager"
if not os.path.exists(MGR):
    !git clone -q https://github.com/ltdrdata/ComfyUI-Manager.git {MGR}

import torch
print("torch:", torch.__version__, "| cuda:", torch.version.cuda)
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print("GPU:", torch.cuda.get_device_name(0), "| capability:", cap)
    if cap[0] >= 12:
        print("✅ Blackwell, fp16 主模型可直接跑, 无需量化")
else:
    print("⚠️ 未检测到 GPU, 请检查运行时类型")

# 验证 Bernini 原生节点是否存在
hit = subprocess.run("grep -ril 'bernini' /content/ComfyUI/comfy_extras/ | head -5",
                     shell=True, capture_output=True, text=True).stdout.strip()
print("✅ 找到 Bernini 节点:\n" + hit if hit else
      "⚠️ 未找到 Bernini 节点, ComfyUI 版本可能太旧, 请删掉 /content/ComfyUI 重跑本格")

print("\n✅ Cell 1 完成")


In [ ]:
# ==========================================
# Cell 2: 下载 Bernini-R 所需的 4 个模型
# 清单来自 ComfyUI 官方文档 docs.comfy.org/tutorials/video/bytedance/bernini-r
# ==========================================
import os
import shutil
from huggingface_hub import hf_hub_download
from concurrent.futures import ThreadPoolExecutor

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("⚠️ 未读到 HF_TOKEN (左侧 🔑 Secrets), 公开仓库仍可下载")

M = "/content/ComfyUI/models"

downloads = [
    # --- 主模型 (fp16 合并版). 如该文件名不存在, 自动回退到 high_noise 分体版 ---
    {"repo": "Comfy-Org/Bernini-R",
     "file": "wan2.2_bernini_r_fp16.safetensors",
     "alt": ["diffusion_models/wan2.2_bernini_r_fp16.safetensors",
             "diffusion_models/wan2.2_bernini_r_high_noise_fp16.safetensors"],
     "dir": M + "/diffusion_models"},

    # --- 文本编码器 ---
    {"repo": "Comfy-Org/Wan_2.1_ComfyUI_repackaged",
     "file": "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
     "dir": M + "/text_encoders"},

    # --- VAE ---
    {"repo": "Kijai/WanVideo_comfy",
     "file": "Wan2_1_VAE_bf16.safetensors",
     "dir": M + "/vae"},

    # --- 加速蒸馏 LoRA (T2V 版, 注意与 SCAIL-2 的 I2V 版不同) ---
    {"repo": "Kijai/WanVideo_comfy",
     "file": "lightx2v_T2V_14B_cfg_step_distill_v2_lora_rank64_bf16.safetensors",
     "dir": M + "/loras"},

    # ==========================================
    # 以下为可选项, 默认不下载
    # ==========================================
    # ❌ 低显存量化版 (fp8 / int8 / mxfp8): G4 显存充足, 用 fp16 画质更好
    # {"repo": "Comfy-Org/Bernini-R", "file": "diffusion_models/wan2.2_bernini_r_high_noise_fp8_scaled.safetensors", "dir": M + "/diffusion_models"},
    # {"repo": "Comfy-Org/Bernini-R", "file": "diffusion_models/wan2.2_bernini_r_low_noise_fp8_scaled.safetensors", "dir": M + "/diffusion_models"},
    # ❌ 1.3B 小模型: 给低配卡应急用
    # {"repo": "Comfy-Org/Bernini-R", "file": "diffusion_models/wan2.1_bernini_1.3B_fp16.safetensors", "dir": M + "/diffusion_models"},
]


def fetch(task):
    os.makedirs(task["dir"], exist_ok=True)
    candidates = [task["file"]] + task.get("alt", [])
    name = os.path.basename(task["file"])
    final = os.path.join(task["dir"], name)
    last = None

    if os.path.exists(final):
        print("⏩ 已存在, 跳过: " + name)
        return

    for cand in candidates:
        try:
            p = hf_hub_download(repo_id=task["repo"], filename=cand, local_dir=task["dir"])
            dst = os.path.join(task["dir"], os.path.basename(cand))
            if os.path.abspath(p) != os.path.abspath(dst):
                shutil.move(p, dst)
            print("✅ 完成: " + os.path.basename(cand))
            return
        except Exception as e:
            last = e
    print("❌ 失败 " + name + ": " + str(last))


print("🚀 开始下载 " + str(len(downloads)) + " 个模型...")
with ThreadPoolExecutor(max_workers=4) as ex:
    list(ex.map(fetch, downloads))

print("\n=== 📁 模型目录清单 ===")
for sub in ["diffusion_models", "text_encoders", "vae", "loras"]:
    d = os.path.join(M, sub)
    if os.path.isdir(d):
        for f in os.listdir(d):
            fp = os.path.join(d, f)
            if os.path.isfile(fp):
                print("  " + sub + "/" + f + "  (" + str(round(os.path.getsize(fp) / 1e9, 2)) + " GB)")

print("\n🎉 模型就绪")


In [ ]:
# ==========================================
# Cell 3: 下载官方工作流模板 (图像编辑 + 视频编辑)
# ==========================================
import os
import urllib.request

wf_dir = "/content/ComfyUI/user/default/workflows"
os.makedirs(wf_dir, exist_ok=True)

BASE = ("https://raw.githubusercontent.com/Comfy-Org/workflow_templates/"
        "main/templates/")

templates = [
    "video_bernini_r_image_editing.json",
    "video_bernini_r_video_editing.json",
]

for t in templates:
    try:
        urllib.request.urlretrieve(BASE + t, os.path.join(wf_dir, t))
        print("✅ 已下载: " + t)
    except Exception as e:
        print("❌ 失败 " + t + ": " + str(e))
        print("   可改用 ComfyUI 内置模板库: 工作流 -> 浏览模板 -> 搜索 Bernini-R")

print("\n👉 启动后在左侧 Workflows 面板直接打开")


In [ ]:
# ==========================================
# Cell 4: FRP 内网穿透配置 (tcp, 无 token)
# ==========================================
import os
import subprocess

FRP_HOST    = "usoren.usdream.dpdns.org"
FRP_PORT    = 7000
REMOTE_PORT = 8092
FRP_VER     = "0.56.0"
FRP_DIR     = "/content/frp_" + FRP_VER + "_linux_amd64"
ACCESS_URL  = "http://" + FRP_HOST + ":" + str(REMOTE_PORT)

if not os.path.exists(FRP_DIR + "/frpc"):
    print("⏳ 下载 frp ...")
    subprocess.run(
        "wget -qO- https://github.com/fatedier/frp/releases/download/v"
        + FRP_VER + "/frp_" + FRP_VER + "_linux_amd64.tar.gz | tar -xz -C /content",
        shell=True)
assert os.path.exists(FRP_DIR + "/frpc"), "frpc 下载失败, 请重跑本格"
subprocess.run("chmod +x " + FRP_DIR + "/frpc", shell=True)

frpc_conf = (
    'serverAddr = "' + FRP_HOST + '"\n'
    'serverPort = ' + str(FRP_PORT) + '\n'
    'loginFailExit = false\n'
    'transport.tcpMux = true\n'
    'transport.poolCount = 5\n'
    'log.to = "/content/frpc.log"\n'
    'log.level = "info"\n\n'
    '[[proxies]]\n'
    'name = "bernini_colab"\n'
    'type = "tcp"\n'
    'localIP = "127.0.0.1"\n'
    'localPort = 8188\n'
    'remotePort = ' + str(REMOTE_PORT) + '\n')

with open(FRP_DIR + "/frpc.toml", "w") as f:
    f.write(frpc_conf)

print("✅ frpc.toml 已写入:")
print("-" * 50)
print(frpc_conf)
print("-" * 50)
print("👉 启动后访问: " + ACCESS_URL)
print("⚠️  地址必须带端口, 不带端口看到的是 frps 自带的 404 页")


In [ ]:
# ==========================================
# Cell 5: 启动 frpc + ComfyUI (重启界面只跑这一格)
# ==========================================
import os
import time
import threading
import subprocess
import configparser

COMFY      = "/content/ComfyUI"
FRP_DIR    = "/content/frp_0.56.0_linux_amd64"
ACCESS_URL = "http://usoren.usdream.dpdns.org:8092"

assert os.path.isdir(COMFY), "找不到 ComfyUI, 请先跑 Cell 1"
assert os.path.exists(FRP_DIR + "/frpc.toml"), "找不到 frpc.toml, 请先跑 Cell 4"
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"


def keep_alive():
    while True:
        time.sleep(300)
        print("\n[Keep-Alive] 保持连接活跃...")


threading.Thread(target=keep_alive, daemon=True).start()

for log in ["/content/comfy.log", "/content/frpc.log"]:
    if os.path.exists(log):
        os.remove(log)

# --- 1. 先起 frpc (秒级) ---
print("⏳ 启动 FRP 穿透...")
subprocess.run("pkill -f '" + FRP_DIR + "/frpc' || true", shell=True)
subprocess.Popen(FRP_DIR + "/frpc -c " + FRP_DIR +
                 "/frpc.toml >> /content/frpc.log 2>&1", shell=True)
time.sleep(6)

frp_log = ""
if os.path.exists("/content/frpc.log"):
    frp_log = open("/content/frpc.log", errors="ignore").read()

if "start proxy success" in frp_log:
    print("✅ FRP 隧道已建立 -> " + ACCESS_URL)
elif "token in login" in frp_log:
    print("❌ 服务端开了 token 验证, 请删掉 frps 的 auth.token 后 systemctl restart frps")
elif "already used" in frp_log:
    print("❌ 远程端口被占用, 请改 Cell 4 的 REMOTE_PORT")
elif "port not allowed" in frp_log:
    print("❌ 端口不在 frps 的 allowPorts 范围内")
else:
    print("⚠️ frpc 日志:")
print(frp_log[-1200:] if frp_log else "(日志为空)")

# --- 2. 关掉 Manager 启动时的联网 Fetch ---
for cfg in [COMFY + "/user/__manager/config.ini",
            COMFY + "/user/default/ComfyUI-Manager/config.ini",
            COMFY + "/custom_nodes/ComfyUI-Manager/config.ini"]:
    os.makedirs(os.path.dirname(cfg), exist_ok=True)
    c = configparser.ConfigParser()
    if os.path.exists(cfg):
        c.read(cfg)
    if "default" not in c:
        c["default"] = {}
    c["default"]["network_mode"] = "private"
    with open(cfg, "w") as f:
        c.write(f)

# --- 3. 启动 ComfyUI ---
LAUNCH = ("python main.py --listen 127.0.0.1 --port 8188 "
          "--enable-cors-header '*' --preview-method auto")
print("\n⏳ 启动 ComfyUI...")
subprocess.Popen(LAUNCH + " > /content/comfy.log 2>&1", shell=True, cwd=COMFY)

ready = False
for i in range(120):
    time.sleep(2)
    if not os.path.exists("/content/comfy.log"):
        continue
    txt = open("/content/comfy.log", errors="ignore").read()
    if "To see the GUI go to" in txt:
        ready = True
        print("✅ ComfyUI ready")
        break
    if "Traceback" in txt and i > 10:
        print("❌ 启动报错:")
        print(txt[-3000:])
        break

if not ready:
    print("--- 日志尾部 ---")
    if os.path.exists("/content/comfy.log"):
        print(open("/content/comfy.log", errors="ignore").read()[-3000:])

print("\n============================================================")
print("🎉 ComfyUI : " + ACCESS_URL)
print("⚠️  地址必须带 :8092")
print("============================================================\n")

subprocess.run("tail -f /content/comfy.log", shell=True)
